In [3]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [4]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [14]:
n = 100

In [15]:
# helper
def quantum_random_bit():
    """Generate a random bit by measuring |+>"""
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    backend = BasicSimulator()
    job = backend.run(transpile(qc, backend), shots=1)
    result = job.result().get_counts()
    return int(list(result.keys())[0])

def quantum_random_bits(n):
    """Generate n random bits quantumly"""
    return [quantum_random_bit() for _ in range(n)]


In [7]:
# alice
alice_bits  = quantum_random_bits(n)
alice_bases = quantum_random_bits(n)

def alice_prepare(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

circuits = [alice_prepare(alice_bits[i], alice_bases[i]) for i in range(n)]
print(f"Alice bits:  {alice_bits[:10]}...")
print(f"Alice bases: {alice_bases[:10]}...")

Alice bits:  [1, 0, 0, 0, 1, 0, 1, 1, 1, 1]...
Alice bases: [0, 0, 1, 0, 0, 0, 1, 0, 1, 0]...  (0=Z, 1=X)


In [16]:
# Attacker Eve

eve_bases = quantum_random_bits(n)   # Eve randomly guesses bases

def eve_intercept(qc, basis):
    """Eve measures in her guessed basis, then resends"""
    qc_copy = qc.copy()
    # Measure in Eve's chosen basis
    if basis == 1:
        qc_copy.h(0)
    qc_copy.measure(0, 0)
    backend = BasicSimulator()
    job = backend.run(transpile(qc_copy, backend), shots=1)
    eve_bit = int(list(job.result().get_counts().keys())[0])

    # Eve resends a new qubit prepared with what she measured
    qc_new = QuantumCircuit(1, 1)
    if eve_bit == 1:
        qc_new.x(0)
    if basis == 1:
        qc_new.h(0)     # resend in same basis Eve used
    return qc_new

# Eve intercepts ALL qubits and resends
intercepted = [eve_intercept(circuits[i], eve_bases[i]) for i in range(n)]
print(f"Eve bases: {eve_bases[:10]}...  (Eve intercepts every qubit)")

Eve bases: [0, 0, 1, 0, 0, 0, 0, 1, 0, 1]...  (Eve intercepts every qubit)


In [17]:
# bob
bob_bases = quantum_random_bits(n)

def bob_measure(qc, basis):
    qc = qc.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    backend = BasicSimulator()
    job = backend.run(transpile(qc, backend), shots=1)
    return int(list(job.result().get_counts().keys())[0])

bob_results = [bob_measure(intercepted[i], bob_bases[i]) for i in range(n)]
print(f"Bob bases:   {bob_bases[:10]}...")
print(f"Bob results: {bob_results[:10]}...")

Bob bases:   [0, 0, 1, 0, 1, 1, 1, 0, 0, 1]...
Bob results: [0, 0, 0, 0, 1, 0, 1, 1, 1, 1]...


In [18]:
# sifting
# Alice and Bob publicly compare bases, keep matching ones
matching = [i for i in range(n) if alice_bases[i] == bob_bases[i]]
alice_key = [alice_bits[i]  for i in matching]
bob_key   = [bob_results[i] for i in matching]
print(f"Matching positions: {len(matching)} out of {n}")

Matching positions: 51 out of 100


In [19]:
# error
# Sacrifice first 10 bits to check for errors
sample_size = min(10, len(alice_key))
errors = sum(alice_key[i] != bob_key[i] for i in range(sample_size))
error_rate = errors / sample_size

print(f"Checking {sample_size} bits...")
print(f"Errors: {errors}, Error rate: {error_rate:.0%}")

# Without Eve: ~0% errors on matching bits
# With Eve:    ~25% errors (Eve guesses wrong basis ~50% of time,
#              and wrong basis causes ~50% chance of error → 25% overall)

THRESHOLD = 0.1   # 10% error rate threshold
if error_rate > THRESHOLD:
    print("HIGH ERROR RATE — possible attacker detected! Abort.")
else:
    print("No attacker detected.")

Checking 10 bits...
Errors: 2, Error rate: 20%
HIGH ERROR RATE — possible attacker detected! Abort.
